In [179]:
import matplotlib.pyplot as plt
from typing import List
import glob
import os
import numpy as np
import pandas as pd 
# import seaborn as sns
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio


import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# Coleta dos dados

In [180]:
results_flows_directories = glob.glob('../../results/results_flows*/*')
results_flows_directories

['../../results/results_flows/hephaestus_s_50_p_6_a_1.0_c_0',
 '../../results/results_flows/darsppo_s_50_p_6_a_1.0_c_0',
 '../../results/results_flows/ga_s_50_p_6_a_1.0_c_0',
 '../../results/results_flows/kuririnPPO_s_50_p_6_a_1.0_c_0']

In [181]:
metricas_de_coleta = ['tempo',
                        'cpu_utilization',
                        'cache_utilization',
                        'bandwidth_utilization',
                        'bandwidth_utilization_per_flow',
                        'success',
                        'latency',
                        'decision_time_ms',
                        'running_sfcs',
                        'cpu_saved',
                        'shared_vnfs',
                        'cpu_per_flow',
                        'cache_per_flow',
                        "total_energy_consumption",
                        "desvio_padrao_latencia_por_sessão",
                        "acceptance_rate",
                        "energy_consumption_per_flow",
                        "queue_time"]

### Função auxiliar que insere a coluna de desvio padrão da latencia nas SFCs de mesma sessão

In [182]:
import pandas as pd
import numpy as np

def calcular_desvio_padrao_por_sessao(df):
    """
    Calcula o desvio padrão da latência por sessão e o adiciona ao DataFrame.

    A sessão é extraída da coluna 'sfc_id' (ex: 'sfc_unique_p5_1' -> sessão '1').
    O desvio padrão de um grupo com apenas um membro é 'NaN', que é
    convertido para 0.

    Argumentos:
    df (pd.DataFrame): O DataFrame de entrada. Deve conter as colunas
                         'sfc_id' e 'latency'.

    Retorna:
    pd.DataFrame: Uma cópia do DataFrame original com a nova coluna
                  'desvio_padrao_latencia_por_sessão'.
    """
    # Criar uma cópia para evitar modificar o DataFrame original (boa prática)
    df_modificado = df.copy()

    # 1. Extrair o ID da sessão da coluna 'sfc_id'
    #    Ex: 'sfc_unique_p5_1' -> '1'
    df_modificado['session_id'] = df_modificado['sfc_id'].str.split('_').str[-1]

    # 2. Calcular o desvio padrão da 'latency' para cada 'session_id'
    #    A função transform() aplica o resultado do cálculo de volta a 
    #    todas as linhas originais do grupo.
    df_modificado['desvio_padrao_latencia_por_sessão'] = df_modificado.groupby('session_id')['latency'].transform('std')

    # 3. Tratar casos de sessão única
    #    Se uma sessão tiver apenas uma entrada, seu desvio padrão será 'NaN'
    #    (Não é possível calcular desvio com um único ponto).
    #    Substituímos esses 'NaN' por 0, pois não há desvio.
    df_modificado['desvio_padrao_latencia_por_sessão'] = df_modificado['desvio_padrao_latencia_por_sessão'].fillna(0)

    # 4. (Opcional) Remover a coluna temporária 'session_id'
    df_modificado = df_modificado.drop(columns=['session_id'])

    return df_modificado



def calcular_consumo_energia_por_fluxo(df):
    """
    Calcula o consumo de energia por fluxo e o adiciona ao DataFrame.

    A nova coluna se chamará 'energy_consumption_per_flow'.
    O cálculo é 'total_energy_consumption' / 'running_sfcs'.

    Casos onde 'running_sfcs' é 0 (o que causaria uma divisão por zero)
    terão o resultado 'energy_consumption_per_flow' definido como 0.0.

    Argumentos:
    df (pd.DataFrame): O DataFrame de entrada. Deve conter as colunas
                         'total_energy_consumption' e 'running_sfcs'.

    Retorna:
    pd.DataFrame: Uma cópia do DataFrame original com a nova coluna
                  'energy_consumption_per_flow'.
    """
    # 1. Criar uma cópia para evitar modificar o DataFrame original
    df_modificado = df.copy()

    # 2. Definir o nome da nova coluna (em inglês, como solicitado)
    nova_coluna = 'energy_consumption_per_flow'
    
    # 3. Calcular a divisão, tratando a divisão por zero
    #    Usamos np.where() como uma forma segura de fazer um 'if/else'
    #    em colunas do Pandas.
    #
    #    Condição: A coluna 'running_sfcs' é igual a 0?
    #    Se sim (True): O valor da nova coluna será 0.0
    #    Se não (False): O valor será a divisão normal
    df_modificado[nova_coluna] = np.where(
        df_modificado['running_sfcs'] == 0,  # Condição
        0.0,                                 # Valor se True
        df_modificado['total_energy_consumption'] / df_modificado['running_sfcs'] # Valor se False
    )

    return df_modificado

def calcular_banda_por_fluxo(df):
    """
    Calcula o consumo de energia por fluxo e o adiciona ao DataFrame.

    A nova coluna se chamará 'energy_consumption_per_flow'.
    O cálculo é 'bandwidth_utilization' / 'running_sfcs'.

    Casos onde 'running_sfcs' é 0 (o que causaria uma divisão por zero)
    terão o resultado 'energy_consumption_per_flow' definido como 0.0.

    Argumentos:
    df (pd.DataFrame): O DataFrame de entrada. Deve conter as colunas
                         'bandwidth_utilization' e 'running_sfcs'.

    Retorna:
    pd.DataFrame: Uma cópia do DataFrame original com a nova coluna
                  'energy_consumption_per_flow'.
    """
    # 1. Criar uma cópia para evitar modificar o DataFrame original
    df_modificado = df.copy()

    # 2. Definir o nome da nova coluna (em inglês, como solicitado)
    nova_coluna = 'bandwidth_utilization_per_flow'
    
    # 3. Calcular a divisão, tratando a divisão por zero
    #    Usamos np.where() como uma forma segura de fazer um 'if/else'
    #    em colunas do Pandas.
    #
    #    Condição: A coluna 'running_sfcs' é igual a 0?
    #    Se sim (True): O valor da nova coluna será 0.0
    #    Se não (False): O valor será a divisão normal
    df_modificado[nova_coluna] = np.where(
        df_modificado['running_sfcs'] == 0,  # Condição
        0.0,                                 # Valor se True
        df_modificado['bandwidth_utilization'] / df_modificado['running_sfcs'] # Valor se False
    )

    return df_modificado

In [183]:
big_data = pd.DataFrame()

def colect_data_from_alg_directory(results_flows_directories):
    data_nla = []

    for alg_dir in results_flows_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split('/')[-1].split('_')
        share = 'y'
        alg_name = simu_exec_name[0].split("\\")[0]
        print(alg_name)
        if alg_name == 'g':
            alg_name = 'Greedy'
        if alg_name == 'ga':
            alg_name =  'GA'
        if alg_name == 'msf':
            alg_name = 'MSF'
        if alg_name == 'goku':
            alg_name = 'OSCIM'
        if alg_name == 'vegeta':
            alg_name = 'Resilient-OSCIM'
        if alg_name == 'musfico':
            alg_name = 'MuSFiCO'
        if alg_name == 'greedyb':
            alg_name = 'GreedyB'
        if alg_name == 'kuririnPPO':
            alg_name = 'Kuririn PPO'
        if alg_name == "darsppo":
            alg_name = "DARSPPO"
        if alg_name == "hephaestus":
            alg_name = "hephaestus"

        files = os.listdir(alg_dir)
        print(f"Simulação:{alg_name}") 
        print("Quantidade de csv: ",len(files))
        
        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
                simulation_df["acceptance_rate"] = simulation_df["acceptance_rate"] /100
                simulation_df = calcular_desvio_padrao_por_sessao(simulation_df)
                simulation_df = calcular_consumo_energia_por_fluxo(simulation_df)
                simulation_df = calcular_banda_por_fluxo(simulation_df)
            except:
                continue
        
            primeiro_tempo = simulation_df['timestamp'].values[0]
            simulation_df['tempo'] = simulation_df[['timestamp']].applymap(lambda x: x - primeiro_tempo)
            simulation_is_success = 'sfc_cache_p4_50' in simulation_df['sfc_id'].values
            
            if simulation_is_success: 
                simulacoes_okays = simulacoes_okays + 1
                simulation_df = simulation_df[metricas_de_coleta]
                
                #Arrendondar tempo
                simulation_df['tempo'] = simulation_df['tempo'].astype(int)
                
                simulation_df = simulation_df.replace('None', pd.NA)

                latency_col = simulation_df[['tempo','latency']] 
                latency_col.dropna(inplace=True)
                latency_col.loc[:, 'latency'] = latency_col['latency'].astype(float)


                ###############################################
                cumulative_sum_success = 0
                cumulative_avg_success = []
                for i, value in enumerate(simulation_df['success']):
                    cumulative_sum_success += value
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df['success'] = cumulative_avg_success
                ###############################################

                simulation_df = simulation_df.groupby('tempo', as_index=False).mean(numeric_only=True)
                latency_df = latency_col.groupby('tempo',as_index=False).mean(numeric_only=True).reset_index()
                
                tempo_range = simulation_df['tempo'].max()
                df_mean = simulation_df.set_index('tempo').reindex(range(tempo_range + 1))
                df_mean = df_mean.fillna(method='ffill')
                df_mean = df_mean.reset_index()


                latency_df = latency_df.set_index('tempo').reindex(range(tempo_range + 1))
                latency_df = latency_df.fillna(method='ffill')
                latency_df = latency_df.reset_index()
         
                df_mean['latency'] = latency_df['latency']
                df_mean['algorithm'] = alg_name
                #df_mean['sharing'] = share
                df_mean= df_mean.iloc[0:1000]
                data_nla.append(df_mean)

        data_nla_f = pd.concat(data_nla)
        print("Simulações de sucesso: ",simulacoes_okays)
        print("Dados Nulos: ", data_nla_f.isnull().sum().sum())
        print()

    return data_nla_f

In [184]:
big_data = colect_data_from_alg_directory(results_flows_directories)

hephaestus
Simulação:hephaestus
Quantidade de csv:  50
Simulações de sucesso:  47
Dados Nulos:  0

darsppo
Simulação:DARSPPO
Quantidade de csv:  33
Simulações de sucesso:  23
Dados Nulos:  0

ga
Simulação:GA
Quantidade de csv:  33
Simulações de sucesso:  33
Dados Nulos:  0

kuririnPPO
Simulação:Kuririn PPO
Quantidade de csv:  33
Simulações de sucesso:  33
Dados Nulos:  0



In [185]:
big_data

,tempo,cpu_utilization,cache_utilization,bandwidth_utilization,bandwidth_utilization_per_flow,success,latency,decision_time_ms,running_sfcs,cpu_saved,shared_vnfs,cpu_per_flow,cache_per_flow,total_energy_consumption,desvio_padrao_latencia_por_sessão,acceptance_rate,energy_consumption_per_flow,queue_time,algorithm
0,0,0.069233,0.037383,0.049950,0.006967,1.000000,14.896667,5.371417,7.0,10.165000,6.666667,9.996139,8.204167,2178.861238,6.373591,1.000000,437.857235,0.060649,hephaestus
1,1,0.069233,0.037383,0.049950,0.006967,1.000000,14.896667,5.371417,7.0,10.165000,6.666667,9.996139,8.204167,2178.861238,6.373591,1.000000,437.857235,0.060649,hephaestus
2,2,0.069233,0.037383,0.049950,0.006967,1.000000,14.896667,5.371417,7.0,10.165000,6.666667,9.996139,8.204167,2178.861238,6.373591,1.000000,437.857235,0.060649,hephaestus
3,3,0.069233,0.037383,0.049950,0.006967,1.000000,14.896667,5.371417,7.0,10.165000,6.666667,9.996139,8.204167,2178.861238,6.373591,1.000000,437.857235,0.060649,hephaestus
4,4,0.069233,0.037383,0.049950,0.006967,1.000000,14.896667,5.371417,7.0,10.165000,6.666667,9.996139,8.204167,2178.861238,6.373591,1.000000,437.857235,0.060649,hephaestus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,0.451000,0.465667,0.276133,0.004453,0.937469,11.916667,5.961500,62.0,53.993333,58.333333,12.586235,9.936593,3055.586871,2.201372,0.937469,49.312285,0.192557,Kuririn PPO
996,996,0.451000,0.465667,0.276133,0.004453,0.937469,11.916667,5.961500,62.0,53.993333,58.333333,12.586235,9.936593,3055.586871,2.201372,0.937469,49.312285,0.192557,Kuririn PPO
997,997,0.455300,0.474000,0.267100,0.004452,0.937713,13.250000,5.876500,60.0,48.660000,56.000000,12.766500,10.450000,3002.393605,2.076094,0.937713,50.039893,0.000144,Kuririn PPO
998,998,0.455300,0.474000,0.267100,0.004452,0.937713,13.250000,5.876500,60.0,48.660000,56.000000,12.766500,10.450000,3002.393605,2.076094,0.937713,50.039893,0.000144,Kuririn PPO


In [186]:
big_data["CPU Salva por SFC"] =  big_data["cpu_saved"] / big_data["running_sfcs"] 
big_data["CPU Salva por Servidor"] =  big_data["cpu_saved"] / 35 

In [187]:
# Suposições
capacidade_maxima_banda_gbps = 10  # Capacidade máxima da banda em Gbps

# Calculando métricas
big_data["eficiencia de cpu"] =  big_data["cpu_utilization"] / big_data["running_sfcs"] 

big_data["eficiencia de banda"] =   big_data["bandwidth_utilization"]   / big_data["running_sfcs"] 
big_data["eficiencia de cache"] =  big_data["cache_utilization"]  / big_data["running_sfcs"] 

# big_data["bit_rate"] = big_data["practical_bandwidth_utilization"] * capacidade_maxima_banda_gbps

# Função para calcular a pontuação da latência
def calcular_pontuacao_latencia(latencia):
    if pd.isna(latencia):
        return 0  # Latência Nula
    elif latencia > 6:
        return -1  # Latência Ruim
    else:
        return 2  # Latência Boa

# Função para calcular a pontuação da aceitação
def calcular_pontuacao_success(success):
    # Convertendo a taxa de sucesso para uma escala de 0 a 1 e multiplicando por 10 para obter uma pontuação máxima de 10
    return success * 10

# Aplicando as funções para calcular as pontuações
big_data['pontuacao_latencia'] = big_data['latency'].apply(calcular_pontuacao_latencia)
big_data['pontuacao_success'] = big_data['success'].apply(calcular_pontuacao_success)

# Calculando a métrica final de qualidade do serviço
big_data['QoS'] = big_data['pontuacao_latencia'] + big_data['pontuacao_success']

In [188]:
# Lista de algoritmos a serem analisados
# algoritmos = ['GreedyB', 'Kuririn PPO']
algoritmos = ['hephaestus', "DARSPPO", 'Kuririn PPO', "GA"]

# Dicionário para armazenar os dados processados de cada algoritmo
dados_processados = {}

def process_data(data):
    data = data.groupby('tempo').mean()
    return data

for alg in algoritmos:
    # Filtrando os dados baseado no algoritmo e na condição de compartilhamento
    dados_filtrados = big_data[(big_data['algorithm'] == alg)]

    # Removendo as colunas 'algor}ithm' e 'sharing'
    dados_filtrados = dados_filtrados.drop(['algorithm'], axis=1)

    # Processando os dados filtrados
    dados_processados[alg] = process_data(dados_filtrados)


In [189]:
dados_processados

{'hephaestus':        cpu_utilization  cache_utilization  bandwidth_utilization  \
 tempo                                                              
 0             0.069914           0.053407               0.059396   
 1             0.069914           0.053407               0.059396   
 2             0.069914           0.053407               0.059396   
 3             0.069914           0.053407               0.059396   
 4             0.069914           0.053407               0.059396   
 ...                ...                ...                    ...   
 995           0.406577           0.432744               0.445246   
 996           0.407356           0.428999               0.444237   
 997           0.408109           0.429892               0.443848   
 998           0.408048           0.432560               0.440776   
 999           0.410844           0.439329               0.443154   
 
        bandwidth_utilization_per_flow   success    latency  decision_time_ms  \
 tempo

In [190]:
# Agora, dados_processados contém os dados processados para cada algoritmo
# Acessando os dados processados para cada algoritmo:
# greedyb_data_share = dados_processados['GreedyB']
kuririnPPO_data_share = dados_processados['Kuririn PPO']
DARSPPO_data_share = dados_processados['DARSPPO']

# res_oscim_data_share = dados_processados['Resilient-OSCIM']

In [191]:
import re
import os
from typing import Dict, Sequence, Optional, Tuple, List
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

def create_boxplot(
    data_series_dict: Dict[str, Sequence],
    yaxis_title: str = 'Y Axis',
    xaxis_title: str = 'Time (s)',
    steps: Optional[int] = 200,
    fill_missing: bool = True,
    bfill_leading: bool = True,
    show: bool = True,
    pdf_filename: Optional[str] = None,
    png_filename: Optional[str] = None,  # <<<<< NOVO PARÂMETRO ADICIONADO
    width: int = 950,
    height: int = 600
) -> Tuple[Optional[go.Figure], pd.DataFrame]:
    """
    Desenha boxplots por janelas para um número arbitrário de séries de dados.

    Args:
        data_series_dict (Dict[str, Sequence]): 
            Um dicionário onde as chaves são os nomes das séries (ex: 'Algoritmo A')
            e os valores são as sequências de dados (listas, np.array, pd.Series).
        yaxis_title (str): Título do eixo Y.
        xaxis_title (str): Título do eixo X.
        steps (int, optional): Tamanho da janela para agrupar os dados no eixo X.
        fill_missing (bool): Se True, preenche valores ausentes (NaN) com ffill.
        bfill_leading (bool): Se True, preenche NaNs iniciais com bfill.
        show (bool): Se True, exibe o gráfico interativamente.
        pdf_filename (str, optional): Nome do arquivo para salvar o PDF.
        png_filename (str, optional): 
            Nome base do arquivo para salvar o PNG (ex: 'meu_grafico.png'). 
            Será salvo automaticamente na pasta 'img_salvas/'.
        width (int): Largura da figura.
        height (int): Altura da figura.

    Returns:
        Tuple[Optional[go.Figure], pd.DataFrame]: 
            A figura do Plotly e o DataFrame usado para a plotagem.
    """

    def _as_series(y: Sequence) -> pd.Series:
        """Converte a entrada para uma pd.Series numérica, coagindo erros."""
        if y is None:
            return pd.Series(dtype="float64")
        s = pd.Series(y, dtype="float64")
        return pd.to_numeric(s, errors="coerce")

    def _window_labels(n: int, step: Optional[int]) -> np.ndarray:
        """Cria rótulos para as janelas de tempo."""
        if n == 0 or step is None or step <= 0:
            return np.arange(n, dtype=int)
        groups = np.repeat(np.arange((n + step - 1) // step), step)[:n]
        return (groups + 1) * step

    # --- Lógica principal ---
    frames: List[pd.DataFrame] = []
    warnings: List[str] = []

    # <<< MUDANÇA PRINCIPAL: Itera sobre o dicionário em vez de argumentos fixos >>>
    for label, data in data_series_dict.items():
        s = _as_series(data)
        if s.empty:
            continue
            
        if fill_missing:
            s = s.ffill()
            if bfill_leading and s.isna().any():
                s = s.bfill()

        if s.dropna().empty:
            warnings.append(f"[AVISO] Série '{label}' ignorada (todos os valores são NaN).")
            continue
            
        s = s.dropna()
        time_labels = _window_labels(len(s), steps)
        
        frames.append(pd.DataFrame({
            xaxis_title: time_labels,
            yaxis_title: s.values,
            "Algorithm": label  # Nome da coluna para a legenda
        }))

    # --- Continuação da lógica de plotagem (sem grandes alterações) ---
    if not frames:
        print("[ERRO] Nenhuma série válida para plotar (vazia ou somente NaN).")
        return None, pd.DataFrame()

    df_plot = pd.concat(frames, ignore_index=True)

    for msg in warnings:
        print(msg)
        
    fig = px.box(df_plot, x=xaxis_title, y=yaxis_title, color="Algorithm")
    fig.update_traces(quartilemethod="exclusive")

    unique_x = sorted(df_plot[xaxis_title].unique())
    fig.update_xaxes(categoryorder="array", categoryarray=unique_x)

    fig.update_layout(
        yaxis=dict(title=yaxis_title, showline=True, showgrid=True),
        xaxis=dict(title=xaxis_title, showline=True, showgrid=True),
        legend_title=None,
        margin=dict(l=100, r=20, b=80, t=40),
        width=width, height=height,
        template="plotly_white",
        legend=dict(x=0.5, y=1.1, xanchor='center', yanchor='bottom', orientation='h'),
        font=dict(family="Arial", size=18, color="Black"),
    )

    if show:
        fig.show()

    # --- Exportação ---
    if pdf_filename:
        try:
            fig.write_image(pdf_filename)
            print(f"Gráfico exportado como PDF: {pdf_filename}")
        except Exception as e:
            html_dir = "HTMLs_Fallback"
            os.makedirs(html_dir, exist_ok=True)
            safe_base = re.sub(r"[^\w\-]+", "_", os.path.splitext(pdf_filename)[0])
            html_out = os.path.join(html_dir, f"{safe_base}.html")
            
            print(f"Falha ao exportar PDF ({e.__class__.__name__}). Salvando em HTML.")
            fig.write_html(html_out, include_plotlyjs="cdn")
            print(f"Gráfico exportado como HTML: {html_out}")

    # <<<<< INÍCIO DA LÓGICA PARA SALVAR PNG <<<<<
    if png_filename:
        output_dir = "img_salvas"
        try:
            # Garante que o diretório "img_salvas" exista
            os.makedirs(output_dir, exist_ok=True)
            
            # Pega apenas o nome base do arquivo, caso o usuário tenha passado um caminho
            base_name = os.path.basename(png_filename)
            
            # Monta o caminho final dentro da pasta "img_salvas"
            save_path = os.path.join(output_dir, base_name)
            
            # Salva a imagem
            fig.write_image(save_path)
            print(f"Gráfico exportado como PNG: {save_path}")
            
        except Exception as e:
            print(f"Falha ao exportar PNG ({e.__class__.__name__}): {e}")
    # <<<<< FIM DA LÓGICA PARA SALVAR PNG <<<<<

    return fig, df_plot

In [192]:
# Dicionário com os títulos das métricas e os nomes das colunas correspondentes
metricas = {
    "CPU Utilization (%)": "cpu_utilization",
    "Cache Utilization (%)": "cache_utilization",
    "Bandwidth Utilization (%)": "bandwidth_utilization",
    "Latency (ms)": "latency",
    "Acceptance Ratio (%)": "acceptance_rate",
    "Decision Time (ms)": "decision_time_ms",
    "Shared SFs": "shared_vnfs",
    "CPU per Flow": 'cpu_per_flow',
    "Cache per Flow": 'cache_per_flow',
    "Bandwidth per Flow":"bandwidth_utilization_per_flow",
    "Energy Consumption (Watts)":"total_energy_consumption",
    "Energy Consumption (Watts) per Flow":"energy_consumption_per_flow",
    "Standard Deviation of Latency per Session": "desvio_padrao_latencia_por_sessão",
    "Queue time (s)": "queue_time"
    
}

# Nomes dos algoritmos para legendas
# A CHAVE deve corresponder ao nome no seu DataFrame `dados_processados`
# O VALOR é o que aparecerá na legenda do gráfico
legendas = {
    # 'GreedyB': 'Greedy Heuristic',
    'Kuririn PPO': 'Inommus',
    'DARSPPO': 'DARSPPO Baseline',
    'hephaestus': 'Hephaestus',

    # Adicione quantos mais algoritmos quiser aqui!
    'GA': 'Genetic Algorithm',
    # 'dp': 'Dynamic Programming'
}

# Iterando sobre cada métrica para plotagem
for titulo_grafico, coluna_df in metricas.items():
    
    # Multiplicador para converter em porcentagem, se necessário
    # (Corrigi a lógica para aplicar o multiplicador corretamente)
    multiplicador = 100 if "%" in titulo_grafico else 1

    # <<< MODO DE USO OTIMIZADO >>>
    # 1. Crie um dicionário para armazenar os dados de todos os algoritmos
    dados_para_plotar = {}
    
    # 2. Preencha o dicionário iterando sobre as legendas
    for alg_id, legenda_nome in legendas.items():
        if alg_id in dados_processados:
            # A legenda (ex: 'Greedy Heuristic') é a chave do dicionário
            # Os dados da coluna correspondente são o valor
            dados_para_plotar[legenda_nome] = dados_processados[alg_id][coluna_df] * multiplicador
        else:
            print(f"[AVISO] Algoritmo '{alg_id}' não encontrado em 'dados_processados'. Pulando.")

    # 3. Chame a função com o dicionário
    if dados_para_plotar: # Apenas plota se houver dados
        
        # <<< MUDANÇAS AQUI >>>
        
        # 1. Define um nome de arquivo único para o PNG
        nome_arquivo_png = f"boxplot_{coluna_df}.png"

        create_boxplot(
            data_series_dict=dados_para_plotar,
            yaxis_title=titulo_grafico,
            
            # 2. Passa o nome do arquivo para o parâmetro png_filename
            # A função cuidará de salvá-lo em "img_salvas/"
            # png_filename=nome_arquivo_png,
            
            # 3. Desativa a exibição interativa para o loop não travar
            show=True 
        )
        # <<< FIM DAS MUDANÇAS >>>

In [193]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from typing import Dict, Optional, Tuple, List
import re
import os

def create_binned_boxplot(
    df_full: pd.DataFrame,
    x_col: str,
    y_col: str,
    alg_col: str,
    legend_map: Dict[str, str],
    bin_size: int = 10,
    x_multiplier: float = 100.0,
    yaxis_title: str = 'Y Axis',
    xaxis_title: str = 'X Axis',
    show: bool = True,
    pdf_filename: Optional[str] = None,
    width: int = 950,
    height: int = 600
) -> Tuple[Optional[go.Figure], pd.DataFrame]:
    """
    Desenha boxplots de y_col vs. x_col, onde x_col é agrupado (binned).

    Args:
        df_full (pd.DataFrame): O DataFrame completo (ex: big_data).
        x_col (str): Nome da coluna para o eixo X (ex: 'cpu_utilization').
        y_col (str): Nome da coluna para o eixo Y (ex: 'latency').
        alg_col (str): Nome da coluna que identifica o algoritmo (ex: 'algorithm').
        legend_map (Dict[str, str]): Dicionário para mapear nomes de alg_col para legendas.
        bin_size (int): O tamanho de cada "caixa" no eixo X (ex: 10).
        x_multiplier (float): Multiplicador para a coluna X (ex: 100.0 para converter 0.08 para 8.0).
        yaxis_title (str): Título do eixo Y.
        xaxis_title (str): Título do eixo X.
        show (bool): Se True, exibe o gráfico.
        pdf_filename (str, optional): Nome do arquivo para salvar o PDF.
        width (int): Largura da figura.
        height (int): Altura da figura.

    Returns:
        Tuple[Optional[go.Figure], pd.DataFrame]: 
            A figura do Plotly e o DataFrame usado para a plotagem.
    """

    # --- Lógica principal ---
    # 1. Criar uma cópia para evitar modificar o original
    df_plot = df_full.copy()

    # 2. Filtrar o DataFrame para incluir apenas os algoritmos no legend_map
    df_plot = df_plot[df_plot[alg_col].isin(legend_map.keys())]
    if df_plot.empty:
        print(f"[ERRO] Nenhum dado encontrado para os algoritmos em 'legend_map'.")
        return None, pd.DataFrame()

    # 3. Mapear os nomes dos algoritmos para os nomes da legenda
    # Cria uma nova coluna 'Algorithm' com os nomes amigáveis para a legenda
    df_plot['Algorithm'] = df_plot[alg_col].map(legend_map)

    # 4. Aplicar o multiplicador à coluna X (ex: 0.08 -> 8.0)
    x_col_scaled = f"{x_col}_scaled"
    df_plot[x_col_scaled] = df_plot[x_col] * x_multiplier

    # 5. Criar os "bins" (agrupamentos) de 10 em 10
    # np.ceil(x / 10) * 10 agrupa (0, 10] como 10, (10, 20] como 20, etc.
    bin_col_label = f"{x_col}_bin"
    df_plot[bin_col_label] = (np.ceil(df_plot[x_col_scaled] / bin_size)) * bin_size
    
    # Converter para inteiro para rótulos mais limpos (ex: "10" em vez de "10.0")
    df_plot[bin_col_label] = df_plot[bin_col_label].astype(int)

    # 6. Ordenar os bins para o eixo X
    unique_x = sorted(df_plot[bin_col_label].unique())

    # --- Plotagem ---
    fig = px.box(
        df_plot,
        x=bin_col_label,  # Eixo X agora são os bins
        y=y_col,          # Eixo Y é a latência
        color="Algorithm", # Cor é o nome do algoritmo mapeado
        points = False
    )
    fig.update_traces(quartilemethod="exclusive")

    # 8. Atualizar eixos 
    fig.update_xaxes(
        title_text=xaxis_title,
        type='category', # Trata os bins como categorias, não números
        categoryorder="array",
        categoryarray=unique_x, # Usa a ordem que calculamos
        
    )

    fig.update_layout(
        yaxis=dict(title=yaxis_title, showline=True, showgrid=True),
        legend_title=None,
        margin=dict(l=100, r=20, b=80, t=40),
        width=width, height=height,
        template="plotly_white",
        legend=dict(x=0.5, y=1.1, xanchor='center', yanchor='bottom', orientation='h'),
        font=dict(family="Arial", size=18, color="Black"),
    )

    if show:
        fig.show()

    # --- Exportação (copiada da sua função original) ---
    if pdf_filename:
        try:
            fig.write_image(pdf_filename)
        except Exception as e:
            html_dir = "HTMLs_Fallback"
            os.makedirs(html_dir, exist_ok=True)
            safe_base = re.sub(r"[^\w\-]+", "_", os.path.splitext(pdf_filename)[0])
            html_out = os.path.join(html_dir, f"{safe_base}.html")
            
            fig.write_html(html_out, include_plotlyjs="cdn")

    return fig, df_plot

In [194]:
# O dicionário de legendas que você já definiu na célula 14
legendas_map = {
    'Kuririn PPO': 'Inommus',
    'DARSPPO': 'DARSPPO Baseline',
    'hephaestus': 'Hephaestus',
    'GA': 'Genetic Algorithm',
}

# Chame a nova função:
metricas = ["latency","acceptance_rate"]

create_binned_boxplot(
    df_full=big_data,
    x_col='cpu_utilization',
    y_col="acceptance_rate",
    alg_col='algorithm',
    legend_map=legendas_map,
    bin_size=10,
    x_multiplier=100.0,  # Converte 0.08 para 8 (para agrupar no bin "10")
    xaxis_title="CPU Utilization (%) Bins (0-10, 10-20, ...)",
    yaxis_title="Acceptance Rate (%)"
)

(Figure({
     'data': [{'alignmentgroup': 'True',
               'boxpoints': False,
               'hovertemplate': ('Algorithm=Hephaestus<br>cpu_ut' ... 'tance_rate=%{y}<extra></extra>'),
               'legendgroup': 'Hephaestus',
               'marker': {'color': '#636efa'},
               'name': 'Hephaestus',
               'notched': False,
               'offsetgroup': 'Hephaestus',
               'orientation': 'v',
               'quartilemethod': 'exclusive',
               'showlegend': True,
               'type': 'box',
               'x': {'bdata': ('CgoKCgoKCgoKCgoKCgoKCgoKChQUFB' ... 'goKCgoKCgoKCgoKCgoKCgoKCgoKCgo'),
                     'dtype': 'i1'},
               'x0': ' ',
               'xaxis': 'x',
               'y': {'bdata': ('AAAAAAAA8D8AAAAAAADwPwAAAAAAAP' ... 'QuT2xl6z9N3OgJiWrrPwPDp2Igcus/'),
                     'dtype': 'f8'},
               'y0': ' ',
               'yaxis': 'y'},
              {'alignmentgroup': 'True',
               'boxpoints'